# Класифікатор дефектів фігур

Донавчаємо готову **MobileNetV2** під нашу
задачу: згорткову базу заморожуємо, а зверху тренуємо власний
класифікатор

In [ ]:
!pip -q install tf2onnx onnx onnxruntime

In [ ]:
import numpy as np
import tensorflow as tf
import keras
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

SEED = 7
IMG_SIZE = 128     
DRAW = 256         
SUPERSAMPLE = 2    
N_SAMPLES = 6000  
BATCH = 64

rng = np.random.default_rng(SEED)
keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)

In [ ]:
SHAPES = ['square', 'rectangle', 'circle', 'ellipse', 'pentagon', 'hexagon']
DEFECTS = ['line', 'bump', 'blob']


def ellipse_outline(rx, ry, steps=96):
    t = np.linspace(0, 2 * np.pi, steps, endpoint=False)
    return list(zip(rx * np.cos(t), ry * np.sin(t)))


def unit_shape(kind, rng):
    if kind == 'square':
        return [(-1, -1), (1, -1), (1, 1), (-1, 1)]
    if kind == 'rectangle':
        b = rng.uniform(0.45, 0.72)
        return [(-1, -b), (1, -b), (1, b), (-1, b)]
    if kind == 'circle':
        return ellipse_outline(1.0, 1.0)
    if kind == 'ellipse':
        return ellipse_outline(1.0, rng.uniform(0.5, 0.78))
    sides = 5 if kind == 'pentagon' else 6
    angles = -np.pi / 2 + np.arange(sides) * 2 * np.pi / sides
    return list(zip(np.cos(angles), np.sin(angles)))


def place(points, scale, rotation, center):
    c, s = np.cos(rotation), np.sin(rotation)
    return [(center + scale * (x * c - y * s), center + scale * (x * s + y * c))
            for x, y in points]


def ray_hit(points, center, angle):
    dx, dy = np.cos(angle), np.sin(angle)
    best = None
    for i in range(len(points)):
        ax, ay = points[i]
        bx, by = points[(i + 1) % len(points)]
        ex, ey = bx - ax, by - ay
        den = ex * dy - dx * ey
        if abs(den) < 1e-9:
            continue
        t = (ex * (ay - center) - ey * (ax - center)) / den
        u = (dx * (ay - center) - dy * (ax - center)) / den
        if t > 0 and 0 <= u <= 1 and (best is None or t < best[0]):
            best = (t, center + dx * t, center + dy * t)
    return best

In [ ]:
def draw_sample(kind, defect, rng):
    size = DRAW * SUPERSAMPLE
    center = size / 2
    image = Image.new('L', (size, size), 255)
    pen = ImageDraw.Draw(image)

    scale = size * 0.28 * rng.uniform(0.85, 1.15)
    points = place(unit_shape(kind, rng), scale, rng.uniform(0, 2 * np.pi), center)
    pen.polygon(points, fill=0)

    if defect is None:
        return image

    angle = rng.uniform(0, 2 * np.pi)
    hit = ray_hit(points, center, angle)
    if hit is None:
        return image

    _, ax, ay = hit
    dx, dy = np.cos(angle), np.sin(angle)
    inside = (ax - dx * scale * 0.06, ay - dy * scale * 0.06)

    if defect == 'line':
        reach = scale * rng.uniform(0.35, 0.70)
        tip = (ax + dx * reach, ay + dy * reach)
        pen.line([inside, tip], fill=0, width=max(2, int(scale * rng.uniform(0.04, 0.09))))

    elif defect == 'bump':
        r = scale * rng.uniform(0.12, 0.22)
        pen.ellipse([ax - r, ay - r, ax + r, ay + r], fill=0)

    else:  # blob
        r = scale * rng.uniform(0.14, 0.22)
        reach = scale * rng.uniform(0.32, 0.55)
        bx, by = ax + dx * reach, ay + dy * reach
        pen.line([inside, (bx, by)], fill=0, width=max(2, int(scale * rng.uniform(0.03, 0.07))))
        pen.ellipse([bx - r, by - r, bx + r, by + r], fill=0)

    return image


def fit_to_frame(image):
    pixels = np.array(image)
    ys, xs = np.where(pixels < 128)
    if len(xs) == 0:
        return None

    crop = image.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    w, h = crop.size
    scale = IMG_SIZE / max(w, h)
    crop = crop.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BILINEAR)

    frame = Image.new('L', (IMG_SIZE, IMG_SIZE), 255)
    frame.paste(crop, ((IMG_SIZE - crop.size[0]) // 2, (IMG_SIZE - crop.size[1]) // 2))
    return np.where(np.array(frame) < 128, 0, 255).astype(np.uint8)


def build_dataset(n):
    images, labels = [], []
    while len(images) < n:
        kind = SHAPES[rng.integers(len(SHAPES))]
        defect = DEFECTS[rng.integers(len(DEFECTS))] if rng.random() < 0.5 else None
        frame = fit_to_frame(draw_sample(kind, defect, rng))
        if frame is None:
            continue
        images.append(np.repeat(frame[:, :, None], 3, axis=2))
        labels.append(1.0 if defect else 0.0)
    return np.array(images, dtype='uint8'), np.array(labels, dtype='float32')


%time X, y = build_dataset(N_SAMPLES)
print('Датасет:', X.shape, '| з дефектом:', int(y.sum()), 'із', len(y))

In [ ]:
order = rng.permutation(len(X))
X, y = X[order], y[order]

n_test = int(len(X) * 0.2)
n_val = int(len(X) * 0.2)
X_test, y_test = X[:n_test], y[:n_test]
X_val, y_val = X[n_test:n_test + n_val], y[n_test:n_test + n_val]
X_train, y_train = X[n_test + n_val:], y[n_test + n_val:]
print('train', len(X_train), '| val', len(X_val), '| test', len(X_test))


def to_dataset(images, labels, shuffle=False):
    data = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        data = data.shuffle(2048, seed=SEED)
    return (data.batch(BATCH)
                .map(lambda batch, label: (tf.cast(batch, tf.float32), label))
                .prefetch(tf.data.AUTOTUNE))


train_ds = to_dataset(X_train, y_train, shuffle=True)
val_ds = to_dataset(X_val, y_val)
test_ds = to_dataset(X_test, y_test)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4.2))
for ax, image, label in zip(axes.ravel(), X_train, y_train):
    ax.imshow(image)
    ax.set_title('дефект' if label else 'норма', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype='float32', name='image')
x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)
x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(1, activation='sigmoid', name='defect')(x)

model = keras.Model(inputs, outputs, name='defect_classifier')
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)
model.summary()

In [ ]:
stop_early = keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=4, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[stop_early])

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)
fine_tune = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[stop_early])

In [ ]:
loss, accuracy, auc = model.evaluate(test_ds, verbose=0)
print(f'test: loss {loss:.4f} | accuracy {accuracy:.4f} | AUC {auc:.4f}')

predicted = (model.predict(test_ds, verbose=0).ravel() > 0.5).astype('float32')
matrix = tf.math.confusion_matrix(y_test, predicted, num_classes=2).numpy()
print('\nМатриця помилок (рядки — істина, стовпці — прогноз):')
print(f'          норма  дефект')
print(f'норма  {matrix[0][0]:7d} {matrix[0][1]:7d}')
print(f'дефект {matrix[1][0]:7d} {matrix[1][1]:7d}')

In [ ]:

for name, metric in [('втрати', 'loss'), ('точність', 'accuracy')]:
    plt.figure(figsize=(6, 3))
    plt.plot(history.history[metric] + fine_tune.history[metric], label='train')
    plt.plot(history.history['val_' + metric] + fine_tune.history['val_' + metric], label='val')
    plt.axvline(len(history.history[metric]) - 0.5, color='gray', ls='--', lw=1)
    plt.title(name)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import shutil
shutil.rmtree('saved_model', ignore_errors=True)

try:
    model.export('saved_model')          # Keras 3
except AttributeError:
    model.save('saved_model', save_format='tf')   # Keras 2

!python -m tf2onnx.convert --saved-model saved_model --output defect.onnx --opset 15

при помилці версії спробувати
`!pip -q install "tf2onnx>=1.16.1" "onnx>=1.16"` і перезапустити середовище

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession('defect.onnx', providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
print('вхід :', input_name, session.get_inputs()[0].shape)
print('вихід:', session.get_outputs()[0].name, session.get_outputs()[0].shape)

sample = X_test[:8].astype('float32')
onnx_probabilities = session.run(None, {input_name: sample})[0].ravel()
keras_probabilities = model.predict(sample, verbose=0).ravel()

print('\nONNX :', np.round(onnx_probabilities, 4))
print('Keras:', np.round(keras_probabilities, 4))
print('максимальна розбіжність:', float(np.abs(onnx_probabilities - keras_probabilities).max()))

In [ ]:
from google.colab import files
files.download('defect.onnx')


Покласти завантажений файл як `models/defect.onnx`

In [ ]:
import onnxruntime.quantization as oq

oq.quantize_dynamic(
    model_input="defect.onnx",
    model_output="defect_quant.onnx",
    weight_type=oq.QuantType.QUInt8
)
files.download('defect_quant.onnx')